# Train Test Split

## Build the Complete Feature Schema

- **Purpose:** Build point-in-time features: use news deduplicated before sentiment inference, align to the first bar end at or after publication, average sentiment per bar, and use exact-time left joins.
- **Input:** News sentiment, one fractional-price feature, and 48 technical indicators.
- **Output:** An in-memory 50-feature candidate schema with missing values retained for later cleaning, plus a preview of the first five rows with all columns displayed.
- **MANIFEST_PATH:** `data/preprocessing/universe/sp500_2025.csv` — Fix the research universe across stages.
- **RESEARCH_START:** `2025-02-01 UTC` — Start research after the preparation period. This also sets the first date in artifact filenames.
- **RESEARCH_END:** `2026-07-01 UTC` — Set the exclusive research end; artifact filenames use the preceding date.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.preprocessing.market_data import ResearchPaths
from src.preprocessing.train_test_split import (
    build_partitioned_candidates,
    build_research_candidates,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]

MANIFEST_PATH = PROJECT_ROOT / "data/preprocessing/universe/sp500_2025.csv"

RESEARCH_START = pd.Timestamp("2025-02-01", tz="UTC")
RESEARCH_END = pd.Timestamp("2026-07-01", tz="UTC")

paths = ResearchPaths(PROJECT_ROOT, start=RESEARCH_START, end=RESEARCH_END)
candidate_path = paths.event("candidates")
event_dir = candidate_path.parent

In [2]:
candidate_schema = build_research_candidates(
    paths, manifest_path=MANIFEST_PATH,
    start=RESEARCH_START, end=RESEARCH_END,
)
with pd.option_context("display.max_columns", None):
    display(candidate_schema.head())

,symbol,event_start,mean_sentiment_score,fractionally_differenced_log_close,On-Balance Volume,Accumulation/Distribution Line,Chaikin Oscillator,Money Flow Index,Williams %R,Aroon Indicator Up,Aroon Indicator Down,Commodity Channel Index,Relative Vigor Index,Force Index,Ultimate Oscillator,Percentage Price Oscillator,Detrended Price Oscillator,Average Directional Index,Chande Momentum Oscillator,Ichimoku Conversion Line,Ichimoku Base Line,Ichimoku Leading Span A,Ichimoku Leading Span B,Stochastic %K,Stochastic %D,MACD Line,MACD Signal Line,Relative Strength Index,Balance of Power,Simple Moving Average (SMA),Exponential Moving Average (EMA),Double Exponential Moving Average (DEMA),TRIX,Triangular Moving Average,Weighted Moving Average (WMA),Hull Moving Average (HMA),Volume Weighted Average Price (VWAP),Parabolic SAR,Pivot Point,Pivot Point Resistance 1,Pivot Point Support 1,Bollinger Band Upper,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
17825,HOLX,2025-02-03 14:30:00.019529+00:00,0.238169,1.566563,-653407.0,-1.263703e+06,-18466.9292,23.7845,-96.1538,57.1429,7.1429,-836.5875,0.0343,-137365.20,-8.9827,-0.4566,0.0157,17.6035,-93.4426,71.290,71.2900,72.1775,72.2325,3.8462,9.6154,-0.1497,-0.0422,3.2787,-0.9600,72.0343,71.9271,71.7199,-0.0082,134.4640,71.9251,71.8266,71.7154,72.2301,72.1300,72.1300,72.1300,72.9467,72.0343,71.1218,1.75,0.1446,72.2164,71.9271,71.6378,72.2000,71.2900,70.38
9531,CRWD,2025-02-03 14:30:00.071916+00:00,0.414413,1.552072,563556.0,8.508266e+05,-11284.4948,21.5279,-92.2902,85.7143,7.1429,-663.7803,0.0130,-824079.40,-7.6985,-0.5821,0.0457,39.1763,-97.5709,392.145,392.2150,398.5542,399.0701,7.7098,2.5699,-1.0463,-0.2708,1.2146,-0.9023,397.6021,396.7735,395.3037,-0.0095,742.1907,396.7537,395.8707,397.8730,399.1023,398.1143,398.1586,398.0257,403.9464,397.6021,391.2579,12.69,1.1066,398.9866,396.7735,394.5603,398.8300,392.2150,385.60
11420,DHR,2025-02-03 14:30:00.380241+00:00,-0.743654,1.406359,-135102.0,-2.056676e+06,468.0391,50.9515,-99.7947,14.2857,7.1429,-690.5446,0.0778,-448847.07,-7.1457,-0.3484,0.1514,82.6995,-83.6215,220.425,220.4250,222.6150,222.5738,0.2053,52.2907,-0.3259,-0.0351,8.1893,-0.9691,222.3607,222.0772,221.5935,-0.0019,415.0733,222.1025,221.9178,222.4684,222.8600,222.7867,222.9033,222.7133,224.8761,222.3607,219.8454,4.85,0.4430,222.9632,222.0772,221.1912,222.8600,220.4250,217.99
29185,PCAR,2025-02-03 14:30:00.565656+00:00,0.877492,4.697841,613358.0,8.172041e+05,-21350.5495,18.5100,-100.0000,85.7143,7.1429,-697.1087,0.0537,-111369.50,-7.4740,-0.2138,-0.0045,15.7666,-91.4729,110.315,110.3200,110.8900,110.9100,0.0000,0.0000,-0.1079,-0.0293,4.2636,-0.9910,110.7978,110.7148,110.5629,-0.0038,206.8225,110.7116,110.6197,110.4727,110.9333,110.8200,110.8300,110.8000,111.4271,110.7978,110.1684,1.11,0.1069,110.9285,110.7148,110.5010,110.9300,110.3200,109.71
20149,JNJ,2025-02-03 14:30:00.766666+00:00,0.516030,1.320920,2419381.0,2.788884e+06,-13960.6110,43.0323,-100.0000,100.0000,7.1429,-421.2855,0.1363,-251257.05,-3.6493,-0.1621,-0.0223,44.7772,-74.7801,151.690,151.7314,152.2394,152.4350,0.0000,34.5997,-0.1336,-0.0869,12.6100,-0.9519,152.0559,152.0154,151.8621,-0.0085,283.8376,151.9968,151.9553,152.0873,152.0500,152.1467,152.1933,152.1133,152.5683,152.0559,151.5434,1.04,0.1443,152.3041,152.0154,151.7268,152.2176,151.6938,151.17


## Create a Test Set

- **Purpose:** Split development and holdout at the fixed calendar boundary, keeping equal timestamps together.
- **Input:** The complete candidate schema and the explicitly configured UTC boundary.
- **Output:** Candidate-event Parquet with partitions and boundary metadata, plus a preview of the first five rows with all columns displayed; rows at or after the boundary belong to sealed holdout.
- **HOLDOUT_BOUNDARY:** `2026-01-01 UTC` — Use February–December 2025 for development and January–June 2026 for holdout.


In [3]:
HOLDOUT_BOUNDARY = pd.Timestamp("2026-01-01", tz="UTC")

In [4]:
development, holdout, candidate_split = build_partitioned_candidates(
    candidate_schema,
    holdout_boundary=HOLDOUT_BOUNDARY,
)

event_dir.mkdir(parents=True, exist_ok=True)
candidate_split.to_parquet(candidate_path, index=False)
print(candidate_path)

holdout_boundary = candidate_split.holdout_boundary.iloc[0]

/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/preprocessing/events/sp500_events_candidates_2025-02-01_2026-06-30.parquet


In [5]:
with pd.option_context("display.max_columns", None):
    display(candidate_split.head())

,symbol,event_start,holdout_boundary,partition,mean_sentiment_score,fractionally_differenced_log_close,On-Balance Volume,Accumulation/Distribution Line,Chaikin Oscillator,Money Flow Index,Williams %R,Aroon Indicator Up,Aroon Indicator Down,Commodity Channel Index,Relative Vigor Index,Force Index,Ultimate Oscillator,Percentage Price Oscillator,Detrended Price Oscillator,Average Directional Index,Chande Momentum Oscillator,Ichimoku Conversion Line,Ichimoku Base Line,Ichimoku Leading Span A,Ichimoku Leading Span B,Stochastic %K,Stochastic %D,MACD Line,MACD Signal Line,Relative Strength Index,Balance of Power,Simple Moving Average (SMA),Exponential Moving Average (EMA),Double Exponential Moving Average (DEMA),TRIX,Triangular Moving Average,Weighted Moving Average (WMA),Hull Moving Average (HMA),Volume Weighted Average Price (VWAP),Parabolic SAR,Pivot Point,Pivot Point Resistance 1,Pivot Point Support 1,Bollinger Band Upper,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
0,HOLX,2025-02-03 14:30:00.019529+00:00,2026-01-01 00:00:00+00:00,development,0.238169,1.566563,-653407.0,-1.263703e+06,-18466.9292,23.7845,-96.1538,57.1429,7.1429,-836.5875,0.0343,-137365.20,-8.9827,-0.4566,0.0157,17.6035,-93.4426,71.290,71.2900,72.1775,72.2325,3.8462,9.6154,-0.1497,-0.0422,3.2787,-0.9600,72.0343,71.9271,71.7199,-0.0082,134.4640,71.9251,71.8266,71.7154,72.2301,72.1300,72.1300,72.1300,72.9467,72.0343,71.1218,1.75,0.1446,72.2164,71.9271,71.6378,72.2000,71.2900,70.38
1,CRWD,2025-02-03 14:30:00.071916+00:00,2026-01-01 00:00:00+00:00,development,0.414413,1.552072,563556.0,8.508266e+05,-11284.4948,21.5279,-92.2902,85.7143,7.1429,-663.7803,0.0130,-824079.40,-7.6985,-0.5821,0.0457,39.1763,-97.5709,392.145,392.2150,398.5542,399.0701,7.7098,2.5699,-1.0463,-0.2708,1.2146,-0.9023,397.6021,396.7735,395.3037,-0.0095,742.1907,396.7537,395.8707,397.8730,399.1023,398.1143,398.1586,398.0257,403.9464,397.6021,391.2579,12.69,1.1066,398.9866,396.7735,394.5603,398.8300,392.2150,385.60
2,DHR,2025-02-03 14:30:00.380241+00:00,2026-01-01 00:00:00+00:00,development,-0.743654,1.406359,-135102.0,-2.056676e+06,468.0391,50.9515,-99.7947,14.2857,7.1429,-690.5446,0.0778,-448847.07,-7.1457,-0.3484,0.1514,82.6995,-83.6215,220.425,220.4250,222.6150,222.5738,0.2053,52.2907,-0.3259,-0.0351,8.1893,-0.9691,222.3607,222.0772,221.5935,-0.0019,415.0733,222.1025,221.9178,222.4684,222.8600,222.7867,222.9033,222.7133,224.8761,222.3607,219.8454,4.85,0.4430,222.9632,222.0772,221.1912,222.8600,220.4250,217.99
3,PCAR,2025-02-03 14:30:00.565656+00:00,2026-01-01 00:00:00+00:00,development,0.877492,4.697841,613358.0,8.172041e+05,-21350.5495,18.5100,-100.0000,85.7143,7.1429,-697.1087,0.0537,-111369.50,-7.4740,-0.2138,-0.0045,15.7666,-91.4729,110.315,110.3200,110.8900,110.9100,0.0000,0.0000,-0.1079,-0.0293,4.2636,-0.9910,110.7978,110.7148,110.5629,-0.0038,206.8225,110.7116,110.6197,110.4727,110.9333,110.8200,110.8300,110.8000,111.4271,110.7978,110.1684,1.11,0.1069,110.9285,110.7148,110.5010,110.9300,110.3200,109.71
4,JNJ,2025-02-03 14:30:00.766666+00:00,2026-01-01 00:00:00+00:00,development,0.516030,1.320920,2419381.0,2.788884e+06,-13960.6110,43.0323,-100.0000,100.0000,7.1429,-421.2855,0.1363,-251257.05,-3.6493,-0.1621,-0.0223,44.7772,-74.7801,151.690,151.7314,152.2394,152.4350,0.0000,34.5997,-0.1336,-0.0869,12.6100,-0.9519,152.0559,152.0154,151.8621,-0.0085,283.8376,151.9968,151.9553,152.0873,152.0500,152.1467,152.1933,152.1133,152.5683,152.0559,151.5434,1.04,0.1443,152.3041,152.0154,151.7268,152.2176,151.6938,151.17
